In [1]:
import io
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from pathlib import Path
from PIL import Image

from albumentations import Compose
from albumentations.augmentations.transforms import *
from albumentations.pytorch.transforms import ToTensorV2 

import torch
from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ModuleNotFoundError: No module named 'albumentations.augmentations.transforms'

In [2]:
batch_size = 32
valid_input_size = (512, 682)
test_img_path = '../input/cassava-leaf-disease-classification/test_images'

In [3]:
model_fnames = [
    '../input/cassava-notebook-15-resnext-models/resnext101wsl_epoch_6.pickle',
    '../input/cassava-notebook-16-models/resnext101wsl_epoch_8.pickle',
]
models = [torch.load(x) 
          for x in model_fnames]
models = [x.to(device) for x in models]
models = [x.eval() for x in models]

NameError: name 'torch' is not defined

In [4]:
test_tfms = Compose([
    Resize(*valid_input_size, always_apply=True),
    Normalize(),
    ToTensorV2(),
])


class TestDataset(Dataset):
    def __init__(self, path, tfms):
        super(TestDataset, self).__init__()
        self.data = [(file.name, open(file, "rb").read())
                     for file in Path(path).iterdir()]
        self.tfms = tfms
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        filename, img_bytes = self.data[idx]
        img = np.array(Image.open(io.BytesIO(img_bytes)).convert('RGB'))
        img = self.tfms(image=img)['image']
        return filename, img
    
test_ds = TestDataset(test_img_path, test_tfms)
test_loader = DataLoader(test_ds, batch_size=batch_size,
                    num_workers=8, drop_last=False)

NameError: name 'Resize' is not defined

In [5]:
class EnsemblePredictor:
    def __init__(self, models):
        super(EnsemblePredictor, self).__init__()
        self.models = models

    def combine_predictions(self, preds):
        result = torch.stack(preds, dim=0).sum(dim=0)
        return [np.argmax(x) for x in result]
    
    def predict_on_loader(self, loader):
        predictions = []
        filenames = []

        for i, (file, img) in enumerate(loader):
            pred = None
            for model in self.models:
                img = img.to(device)
                if pred is None:
                    with torch.no_grad():
                        pred = model(img).cpu().numpy()
                else:
                    with torch.no_grad():
                        pred += model(img).cpu().numpy()
            predictions += [np.argmax(x) for x in pred]
            filenames += file

        return predictions, filenames

In [6]:
predictor = EnsemblePredictor(models)
predictions, filenames = predictor.predict_on_loader(test_loader)

NameError: name 'models' is not defined

In [7]:
with open('submission.csv', 'w+') as submission:
    submission.write('image_id,label\n')
    for prediction, filename in zip(predictions, filenames):
        submission.write(f'{filename},{prediction}\n')

NameError: name 'predictions' is not defined